# Morphology by Condition: Frame 1 vs. Frame 13

Compares cell/nuclear morphology across all four conditions
(`naive`, `vem_1uM`, `gem_2.5nM`, `tram_6nM`) at two independent
timepoints: frame 1 and frame 13 (the last frame any condition's
metadata can contain - see config.py). Each frame has its own,
completely separate CAJAL run (`main.py` never pools masks across
frames), so frame 1's embedding space has no relationship to frame
13's - this notebook never compares a frame to another frame, only
conditions to each other WITHIN a frame.

Requires `main.py` to have already been run (for each frame's own
`selected_metadata.csv` and `{cell,nuc}_{umap,phate}.npz` under
`output/cajal_output/frame_NN/`).


In [ ]:
import os
import sys

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/03_multi_drug_comparison/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/MASKS_DIR/CAJAL_OUTPUT_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here (frame_cajal_dir reads CAJAL_OUTPUT_DIR
# dynamically, so this must happen before it's ever called)
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset
from shared.density_plots import export_kde


## Method

For each frame in `config.FRAMES` (1, 13), each modality (cell, nucleus),
each embedding method (UMAP, PHATE):

1. **Load that frame's own embedding** (`load_full_dataset`, pointed at
   `config.frame_cajal_dir(frame)`) - already exactly the right data,
   since `main.py` ran CAJAL/UMAP/PHATE separately per frame.
2. **Plot**: one KDE contour per condition over that frame's embedding,
   plain grey background scatter (`export_kde`) - a single standalone
   plot, not paged across frames like the other analyses' notebooks,
   since there's no shared embedding here to page through.

2 frames x 2 modalities x 2 embedding methods = 8 plots total.


In [ ]:
CONDITION_COLORS = {
    'naive':     'tab:blue',
    'vem_1uM':   'tab:red',
    'gem_2.5nM': 'tab:orange',
    'tram_6nM':  'tab:green',
}
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_kde_by_condition')

# KDE contour levels (level p encloses the densest (1-p) fraction of a
# group's mass - higher is tighter/closer to the peak) and bandwidth
# multiplier (above 1 smooths over sampling/distance-tie noise)
KDE_LEVELS     = (0.7, 0.8, 0.9)
KDE_BW_ADJUST  = 1

# main.py computes both of these for every (frame, modality) - plot both
# every time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_frame(frame):
    '''
    For each modality and embedding method, load this frame's own
    embedding and plot a KDE contour per condition.

    Inputs:
        frame: frame number (must be one of config.FRAMES)
    Outputs:
        dfs: {(modality, embedding_method): the full merged dataset for
              that combination} (in-memory only)
    '''
    dfs = {}
    for modality in ['cell', 'nuc']:
        for embedding_method in EMBEDDING_METHODS:
            df = load_full_dataset(config.frame_cajal_dir(frame), modality, embedding_method)
            print(f"[frame {frame}/{modality}/{embedding_method}] {len(df)} cells across "
                  f"{df['condition'].nunique()} conditions")

            file_prefix = f'{modality}_frame{frame:02d}_{embedding_method}'
            export_kde(
                df, 'condition', CONDITION_COLORS, PLOTS_DIR, file_prefix, embedding_method.upper(),
                title=f'{MODALITY_NAMES[modality]} morphology: frame {frame}',
                levels=KDE_LEVELS, bw_adjust=KDE_BW_ADJUST,
            )
            dfs[(modality, embedding_method)] = df

    return dfs


## Run: frame 1

In [ ]:
frame1_dfs = process_frame(config.FRAMES[0])


## Run: frame 13

In [ ]:
frame13_dfs = process_frame(config.FRAMES[1])
